### Exploration
In this notebook, we'll explore the current limitations of tiny language models for mathematical reasoning. We'll start by experimenting with a 0.5B parameter model, running it on a variety of math questions to better understand its level of mathematical ability, discover its weaknesses, and analyze where it succeeds and where it fails.

#### Loading the model

In [1]:
# Setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from transformers import AutoModelForCausalLM, AutoTokenizer

model_str = "Qwen/Qwen3-1.7B-Base"  # base model

tokenizer = AutoTokenizer.from_pretrained(model_str)
model = AutoModelForCausalLM.from_pretrained(model_str, dtype="auto", device_map="auto")

/courses/TDDE09/tdde09/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 310/310 [00:00<00:00, 672.66it/s, Materializing param=model.norm.weight]                              


### Below is the format used in the [**DeepSeek R1 paper**](https://arxiv.org/abs/2501.12948)

<div style="font-size: 0.85em; max-width: 750px; line-height: 1.5;">

---
*A conversation between User and Assistant. The user asks a question, and the Assistant solves 
it. The assistant first thinks about the reasoning process in the mind and then provides the user 
with the answer. The reasoning process and answer are enclosed within \<think>...\</think> 
and \<answer>...\</answer> tags, respectively, i.e., \<think> reasoning process here \</think> 
\<answer> answer here \</answer>. User: <span style="color:red">prompt</span>. Assistant:*

---

</div>

We are working with a base model, which is why we have to specify that this is a conversation between a User and Assistant, as well as have the ending be **Assistant:**, because the model will try to predict what an assistant would respond in such a scenario. 

We can create a function for generating this "system prompt" for an arbitrary question, by replacing the <span style="color:red">prompt</span>.

In [2]:
def generate_prompt(question, helper="", think_start_tok="<think>", think_stop_tok="</think>", answer_start_tok="<answer>", answer_stop_tok="</answer>"):
    """
    Wraps a question into the DeepSeek-R1 paper prompt format.
    """

    prompt = (
        "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. "
        "The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. "
        f"The reasoning process and answer are enclosed within {think_start_tok}...{think_stop_tok} and {answer_start_tok}...{answer_stop_tok} tags, "
        f"respectively, i.e., {think_start_tok} reasoning process here {think_stop_tok} {answer_start_tok} answer here {answer_stop_tok}. "
        f"User: {question}. Assistant: {helper}"
    )
    return prompt

We can now use the model with the "system" prompt, and try to get it to answer a math question.

Most likely, the model will completely fail at using the specified tags like it should, even with the added `\<think>` helper token we added.

In [3]:
from utils import lmprint

question = "what is 14 times 5670?"

inputs = tokenizer(generate_prompt(question, "<think>"), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

output = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=0.8,        # Adds randomness
                top_p=0.9,              
                pad_token_id=tokenizer.eos_token_id
            )

generated_tokens = output[0][input_len:]

raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)
raw_full = tokenizer.decode(output[0], skip_special_tokens=False)
lmprint.pretty_print("<think>" + raw_output)
print(f"Raw output: {raw_full}")

╭─ 📝 Response ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│   reasoning process here  expensive to do this calculation by hand, so we will use a calculator to help us.     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ 💡 Answer ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  answer here                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ 📝 Response ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  expensive to do this calculation by hand, so we will use a calculator to help us.  expensive to do this        │
│  calculation by hand, so we will use a calculator to help us.  expensive to do this calculation by hand, so we  │
│  will use a calculator to help us.  expensive to do this calculation by hand, so we will use a calculator to    │
│  help us.  expensive to do this calculation by hand, so we will use a calculator to help us.  expensive to do   │
│  this calculation by hand, so we will use a calculator to help us.  expensive to do this calculation by hand,   │
│  so we will use a calculator to help us.  expensive to do this calculation by hand, so we will use a            │
│  calculator to help us.  expensive to do this calculation by hand, so we will use a calculator to help us.      │
│  expensive to do this calculation by hand, so we will use a calculator to help us.  expensive to do this        │
│  calculation by hand, so we will use a calculator to help us.  expensive to do this calculation by hand, so we  │
│  will use a calculator to help                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw output: A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. User: what is 14 times 5670?. Assistant: <think> reasoning process here  expensive to do this calculation by hand, so we will use a calculator to help us. <answer>  answer here </answer>
 expensive to do this calculation by hand, so we will use a calculator to help us.  expensive to do this calculation by hand, so we will use a calculator to help us.  expensive to do this calculation by hand, so we will use a calculator to help us.  expensive to do this calculation by hand, so we will use a calculator to help us.  expensive to do this calculation by hand, so we will use a calcu

### Why this happens:
Since we are using a base model, this means it has **only** been pretrained.
Our tokenizer has many added tokens, which the base model has never seen, for example `<think>` and `</think>`.
This means that the embeddings for those new tokens are just random and dont have any meaning. 
This is not the case thought for `<answer>`, which is why it manages to output it correctly (most of the time)

In [4]:
# Check if these are single tokens or get split into sub-tokens
test_strings = ["<think>", "</think>", "<answer>", "</answer>"]

for s in test_strings:
    tokens = tokenizer.tokenize(s)
    ids = tokenizer.encode(s, add_special_tokens=False)
    print(f"{s:<15} -> tokens: {tokens}, ids: {ids}")


<think>         -> tokens: ['<think>'], ids: [151667]
</think>        -> tokens: ['</think>'], ids: [151668]
<answer>        -> tokens: ['<', 'answer', '>'], ids: [27, 9217, 29]
</answer>       -> tokens: ['</', 'answer', '>'], ids: [522, 9217, 29]


If the model cannot even follow along with the tags that we specified, perhaps it will not be possible to even get it to start improving, because it will always get bad rewards, and never be able to start improving.

Lets try to give our model some chances at this, because it only has to do it a couple of times for each rollout, since we are going to be doing multiple generations per question. 
Eventually it should learn to do this very consistently.

In [5]:
from utils import checks

question = "what is 14 times 5670?"
helper = "<think>"
prompt = generate_prompt(question, helper=helper)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]
num_tries = 16



inputs = tokenizer(generate_prompt(question, helper), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

outputs = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=1,        # Adds randomness
                num_return_sequences=num_tries, # This does the parallel work for n tries
                pad_token_id=tokenizer.eos_token_id
            )

answer_count = 0
think_count = 0
for i in range(num_tries):
    generated_tokens = outputs[i][input_len:]
    raw_response = helper + tokenizer.decode(generated_tokens, skip_special_tokens=True) # Add back helper

    if checks.has_complete_answer_block(raw_response): answer_count += 1
    if checks.has_complete_thinking_block(raw_response): think_count += 1
    
    if checks.is_format_correct(raw_response):
        lmprint.pretty_print(raw_response)

print(f"Thinking: {(think_count/num_tries)*100:.1f}% of tries.\n",
      f"Answer: {(answer_count/num_tries)*100:.1f}% of tries")
    

Thinking: 0.0% of tries.
 Answer: 50.0% of tries


---

So our model uses the correct thinking tokens 0% of all the generated outputs, so we wont ever be actually learning the correct formatting if this is the case.

--- 

### Possible fixes.
1. Average the new token from the subcomponents that make it up.
    * Set the token `<think>` to the average of the tokens `<`, `think`, `>`.
    * Set the token `</think>` to the average of the tokens `</`, `think`, `>`.
    * This did not work well enough to actually start generating full `<think>` tokens, but instead just closed the thinking with `>`
2. Tiny SFT dataset with formatting data
    * Train model on generated examples where the model uses the think and answer tokens.
    * Ex: "User: What is 2*13? Assistant: \<think>2\*13=26\</think>\<answer>26\</answer>


In [6]:
from data.generate import RandomFormatSFTDataset

SFTdataset = RandomFormatSFTDataset(tokenizer=tokenizer, prompt_template=generate_prompt, size=200)
print("Input text:")
print(tokenizer.decode(SFTdataset[0]["input_ids"]))

Input text:
A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. User: Calculate: (8 + 2) * 9. Assistant: 
<think>
First: 8 + 2 = 10. Then: 10 * 9 = 90.
</think>
<answer>90</answer>


In [ ]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch

# TRAINING CONFIG
EPOCHS      = 3
BATCH_SIZE  = 16
LR          = 1e-3 

# FREEZE MODEL PARAMETERS
for param in model.parameters():
    param.requires_grad = False

# UNFREEZE EMBEDDING AND HEAD LAYER
model.get_input_embeddings().weight.requires_grad = True
model.lm_head.weight.requires_grad = True
trainable_params = [param for param in model.parameters() if param.requires_grad]
print(f"Trainable parameters: {sum(p.numel() for p in trainable_params)}")

optimizer = AdamW(trainable_params, lr=LR)

# Pad a training batch so that everyting is same length
def collate_fn(batch):
    eos_id = tokenizer.eos_token_id
    pad_id = tokenizer.pad_token_id
    # Append EOS to each sample
    new_batch = []
    for b in batch:
        new_batch.append({
            "input_ids": torch.cat([b["input_ids"], torch.tensor([eos_id])]),
            "labels": torch.cat([b["labels"], torch.tensor([eos_id])]),
            "attention_mask": torch.cat([b["attention_mask"], torch.ones(1)]),
        })
    
    max_len = max(b["input_ids"].shape[0] for b in new_batch)
    
    input_ids = []
    labels = []
    attention_mask = []
    
    for b in new_batch:
        pad_len = max_len - b["input_ids"].shape[0]
        input_ids.append(torch.cat([b["input_ids"], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        labels.append(torch.cat([b["labels"], torch.full((pad_len,), -100, dtype=torch.long)]))
        attention_mask.append(torch.cat([b["attention_mask"], torch.zeros(pad_len)]))
    
    return {
        "input_ids": torch.stack(input_ids),
        "labels": torch.stack(labels),
        "attention_mask": torch.stack(attention_mask),
    }

dataloader = DataLoader(SFTdataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

# TRAINING LOOP
model.train()
step = 0

for epoch in range(EPOCHS):
    epoch_loss = 0
    num_batches = 0

    for batch in dataloader:
        # Move to device
        batch = {k: v.to(model.device) for k, v in batch.items()}

        # Forward
        outputs = model(**batch)
        loss = outputs.loss

        # Backward
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        epoch_loss += loss.item()
        num_batches += 1
        step += 1

        if step % 10 == 0:
            print(f"  Step {step:>4d} | Loss: {loss.item():.4f}")

    avg_loss = epoch_loss / num_batches
    print(f"Epoch {epoch+1:>2d}/{EPOCHS} | Avg Loss: {avg_loss:.4f}")


Trainable parameters: 311164928
  Step   10 | Loss: 0.0123
Epoch  1/3 | Avg Loss: 0.2237
  Step   20 | Loss: 0.0050
Epoch  2/3 | Avg Loss: 0.0081
  Step   30 | Loss: 0.0023
Epoch  3/3 | Avg Loss: 0.0020


### After training
We have now trained our model on some examples where the model makes use of the thinking tags. We can now test if it has learned to use the new token.

In [13]:
question = "What is (100 - 12) * 2 - 40?"

inputs = tokenizer(generate_prompt(question, "<think>"), return_tensors="pt").to(model.device)
input_len = inputs.input_ids.shape[1]

output = model.generate(
                **inputs, 
                max_new_tokens=256,    # High, but just a demonstration
                do_sample=True,         # Enable sampling
                temperature=1,        # Adds randomness
                top_p=0.9,              
                pad_token_id=tokenizer.eos_token_id
            )

generated_tokens = output[0][input_len:]

raw_output = tokenizer.decode(generated_tokens, skip_special_tokens=False)
raw_full = tokenizer.decode(output[0], skip_special_tokens=False)
lmprint.pretty_print("<think>" + raw_output)
print(f"Raw output: {raw_full}")

╭─ 🤔 Thinking ───────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  First: 100 - 12 = 88. Then: 88 * 2 = 176.                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ 💡 Answer ─────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  176                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw output: A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning process and answer are enclosed within <think>...</think> and <answer>...</answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>. User: What is (100 - 12) * 2 - 40?. Assistant: <think>
First: 100 - 12 = 88. Then: 88 * 2 = 176. </think>
<answer>176</answer><|endoftext|>
